# BNO055 binary file explorer

Poke at the per-stream `.bin` files written for the BNO055 IMU under one NeuropixelsV2 recording.
Each stream is its own file (no embedded timestamps); timing lives in the `Clock` / `HubSyncCounter` files.
Here we look at the `_84` (sleep) block.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

data_dir = Path("/Users/willreith/Documents/phd/aeon/data/ephys/abcEphysPilot04/26-05-19T14-13-54/NeuropixelsV2")
block = "84"

for p in sorted(data_dir.glob(f"*Bno055*_{block}.bin")):
    print(f"{p.name:45s} {p.stat().st_size:>10d} bytes")

In [ ]:
# stream -> (dtype, n components per sample), inferred from file sizes + content
streams = {
    "Clock":              ("<u8", 1),
    "HubSyncCounter":     ("<u8", 1),
    "Euler":              ("<f4", 3),
    "Quaternion":         ("<f4", 4),
    "GravityVector":      ("<f4", 3),
    "LinearAcceleration": ("<f4", 3),
}

def load(name):
    dtype, ncomp = streams[name]
    arr = np.fromfile(data_dir / f"NeuropixelsV2_Bno055_{name}_{block}.bin", dtype=dtype)
    return arr.reshape(-1, ncomp)

for name in streams:
    a = load(name)
    print(f"{name:20s} shape={str(a.shape):12s} dtype={a.dtype}  head={a[0]}")

In [ ]:
# Clock is a free-running ONIX hardware counter @ 250 MHz
HW_CLOCK_HZ = 250e6

clk = load("Clock").ravel()
dt = np.diff(clk)
t = (clk - clk[0]) / HW_CLOCK_HZ

print("n samples         :", clk.size)
print("duration (s)      :", t[-1])
print("median dt (counts):", np.median(dt))
print("effective rate Hz :", HW_CLOCK_HZ / np.median(dt))
print("HubSyncCounter head:", load("HubSyncCounter").ravel()[:5])

# physical sanity checks
quat = load("Quaternion")
grav = load("GravityVector")
print("quat norm  mean/std:", np.linalg.norm(quat, axis=1).mean(), np.linalg.norm(quat, axis=1).std())
print("|gravity|  mean    :", np.linalg.norm(grav, axis=1).mean())
print("Euler quant step   :", np.min(np.diff(np.unique(load("Euler")[:, 0]))))

In [ ]:
# fusion outputs (Euler/Quaternion/Gravity) update slower than they are sampled;
# linear acceleration changes essentially every sample
def frac_repeated(a):
    return np.all(a[1:] == a[:-1], axis=1).mean()

for name in ["Euler", "Quaternion", "GravityVector", "LinearAcceleration"]:
    print(f"{name:20s} frac identical-to-prev: {frac_repeated(load(name)):.3f}")

In [ ]:
euler = load("Euler")
lin = load("LinearAcceleration")

fig, ax = plt.subplots(3, 1, sharex=True, figsize=(11, 7))
for i, lbl in enumerate(["heading", "roll", "pitch"]):
    ax[0].plot(t, euler[:, i], lw=0.6, label=lbl)
ax[0].set_ylabel("Euler (deg)"); ax[0].legend(loc="upper right", ncol=3)
for i, lbl in enumerate(["x", "y", "z"]):
    ax[1].plot(t, lin[:, i], lw=0.6, label=lbl)
ax[1].set_ylabel("lin accel (m/s\u00b2)"); ax[1].legend(loc="upper right", ncol=3)
for i, lbl in enumerate(["x", "y", "z"]):
    ax[2].plot(t, grav[:, i], lw=0.6, label=lbl)
ax[2].set_ylabel("gravity (m/s\u00b2)"); ax[2].set_xlabel("time (s)"); ax[2].legend(loc="upper right", ncol=3)
fig.tight_layout()

## Speed

**Translational** = high-pass the gravity-removed `LinearAcceleration` (0.3 Hz), integrate → velocity, high-pass again to drop integration drift → ‖v‖ (m/s). Accel-only single integration drifts badly — without the high-pass the speed ramps to ~12 m/s; with it, speed stays ~0 when the animal is still (~93 % of samples < 1 cm/s here) and rises only during movement. Still only a rough proxy, not calibrated ground speed.

**Angular** = rotation angle between consecutive `Quaternion` samples / dt (deg/s). No integration, no drift.

In [ ]:
from scipy.integrate import cumulative_trapezoid
from scipy.signal import butter, filtfilt

fs = 1 / np.median(np.diff(t))                          # effective IMU rate (~118 Hz)
dt_s = np.diff(clk) / HW_CLOCK_HZ                        # per-sample interval (s)

# --- translational speed ---
# accel-only single integration drifts, so high-pass out the bias/slow drift both
# before and after integrating; leaves transient movement, returns to ~0 when still.
def highpass(x, fc=0.3, order=2):
    b, a = butter(order, fc / (fs / 2), btype="high")
    return filtfilt(b, a, x, axis=0)

vel = cumulative_trapezoid(highpass(lin), x=t, axis=0, initial=0)
speed_trans = np.linalg.norm(highpass(vel), axis=1)     # m/s (rough proxy, accel-only)

# --- angular speed: rotation between consecutive unit quaternions / dt ---
q = quat / np.linalg.norm(quat, axis=1, keepdims=True)
cos_half = np.abs(np.sum(q[1:] * q[:-1], axis=1)).clip(0, 1)
ang_speed = np.degrees(2 * np.arccos(cos_half)) / dt_s  # deg/s

fig, ax = plt.subplots(2, 1, sharex=True, figsize=(11, 5))
ax[0].plot(t, speed_trans, lw=0.6)
ax[0].set_ylabel("speed (m/s)"); ax[0].set_title("translational (high-pass ∫ linear accel, 0.3 Hz)")
ax[1].plot(t[1:], ang_speed, lw=0.6, color="C1")
ax[1].set_ylabel("ang. speed (deg/s)"); ax[1].set_xlabel("time (s)"); ax[1].set_title("angular (Δquaternion)")
fig.tight_layout()